# 05 Modeling & LOBO Validation

This notebook orchestrates Stage 05 of the PHM Bearing RUL pipeline.

Pipeline
--------
```

04 Model-ready Learning Dataset + Prefix Definition
↓
Feature / Target Schema Resolution
↓
ModelingConfig
↓
LOBO Model Training
↓
ModelBundle
├── Prefix Validation Prediction
│       ↓
│   RUL Reconstruction
│       ↓
│   Evaluation
│
└── Full Validation Prediction
↓
RUL Reconstruction
↓
RUL Trajectory Visualization

Experimental Calibration
↓
Raw / Calibrated Evaluation
↓
Calibration Trajectory Comparison

```

Stage 05 principles
-------------------
- One model is trained for each held-out bearing.
- Each LOBO model is trained using the configured TrainingRange
  from the remaining bearings.
- Prefix definitions select validation samples for official evaluation.
- One prefix definition selects exactly one validation row.
- Each validation row is predicted only by the model that held out
  its bearing.
- Full validation prediction covers the complete lifetime of each
  held-out bearing and is used only for visualization.
- Prefix Definition and TrainingRange do not restrict visualization.
- No ensemble is performed in Stage 05.
- The prediction target is RUL_norm.

- Calibration is experimental post-processing and does not modify the ModelBundle or the Stage 06 interface.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import json
import joblib
import matplotlib.pyplot as plt

from utils.bearing_utils import (
    reconstruct_rul_sec_from_norm, META_COLUMNS
)
from utils.model_utils import *
from utils.eval_utils import build_3tier_evaluation_report, evaluate_predictions
from utils.eda_utils import plot_rul_prediction
from utils.project_utils import assert_required_columns

In [ ]:
LEARNING_DATASET_PATH = "outputs/04_dataset_builder/model_ready_learning_dataset.csv"
PREFIX_DEFINITION_PATH = "outputs/04_dataset_builder/prefix_definition.csv"
SELECTION_MANIFEST_PATH = "outputs/03_eda_selection/selection_config.json"

print(f"Learning dataset: {LEARNING_DATASET_PATH}")
print(f"Prefix definition: {PREFIX_DEFINITION_PATH}")

learning_df = pd.read_csv(LEARNING_DATASET_PATH)
prefix_definition = pd.read_csv(PREFIX_DEFINITION_PATH)

print("Learning dataset")
print(f"Shape: {learning_df.shape}")

display(learning_df.head())

print("\nPrefix definition")
print(f"Shape: {prefix_definition.shape}")

display(prefix_definition.head())

In [ ]:
TARGET_COLUMN = "RUL_norm"

assert_required_columns(learning_df, [*META_COLUMNS, TARGET_COLUMN])
assert_required_columns(prefix_definition, ["bearing", "end_idx"])

if learning_df.empty:
    raise ValueError("model_ready_learning_dataset must not be empty.")

if prefix_definition.empty:
    raise ValueError("prefix_definition must not be empty.")

print("Input schema validation passed.")

In [ ]:
# =============================================================================
# 5. Load Selection Manifest & Resolve Candidate Features
# =============================================================================

with open(SELECTION_MANIFEST_PATH, "r", encoding="utf-8") as f:
    selection_manifest = json.load(f)

if "candidate_features" not in selection_manifest:
    raise KeyError(
        "Selection manifest must contain 'candidate_features'."
    )

if "selected_target" not in selection_manifest:
    raise KeyError(
        "Selection manifest must contain 'selected_target'."
    )

CANDIDATE_FEATURES = selection_manifest["candidate_features"]
SELECTED_TARGET = selection_manifest["selected_target"]

if not isinstance(CANDIDATE_FEATURES, list):
    raise TypeError("'candidate_features' must be a list.")

if not isinstance(SELECTED_TARGET, str) or not SELECTED_TARGET:
    raise TypeError("'selected_target' must be a non-empty string.")

if SELECTED_TARGET != TARGET_COLUMN:
    raise ValueError(
        "Unexpected target mismatch: "
        f"expected {TARGET_COLUMN!r}, got {SELECTED_TARGET!r}."
    )

missing_candidate_features = [
    feature for feature in CANDIDATE_FEATURES
    if feature not in learning_df.columns
]

if missing_candidate_features:
    raise KeyError(
        "Candidate feature(s) not found in model-ready dataset: "
        f"{missing_candidate_features}"
    )

print(f"Candidate Features : {len(CANDIDATE_FEATURES)}")
for feature in CANDIDATE_FEATURES:
    print(f"  - {feature}")

print(f"Selected Target     : {SELECTED_TARGET}")

In [ ]:
# =============================================================================
# Final Feature Selection
# =============================================================================

SELECTED_FEATURE_COLUMNS = [
    "RDI_cf",
    "RDI_kurt",
    "Rectified_rms",
    "Rectified_p2p",
    "temp_mean",
]

missing_features = [
    feature for feature in SELECTED_FEATURE_COLUMNS
    if feature not in CANDIDATE_FEATURES
]

if missing_features:
    raise KeyError(
        "Selected feature(s) are not included "
        f"in the candidate feature set: {missing_features}"
    )

if not SELECTED_FEATURE_COLUMNS:
    raise ValueError(
        "SELECTED_FEATURE_COLUMNS must contain at least one feature."
    )

if len(SELECTED_FEATURE_COLUMNS) != len(set(SELECTED_FEATURE_COLUMNS)):
    raise ValueError("SELECTED_FEATURE_COLUMNS contains duplicates.")

FEATURE_COLUMNS = list(SELECTED_FEATURE_COLUMNS)

print(f"Candidate features : {len(CANDIDATE_FEATURES)}")
print(f"Selected features  : {len(FEATURE_COLUMNS)}")

print("\nSelected feature columns:")
for feature in FEATURE_COLUMNS:
    print(f"- {feature}")

In [ ]:
training_range = TrainingRange(
    start_ratio=0.3, end_ratio=1.0
)

print("Training range")
print(f"Start ratio: {training_range.start_ratio}")
print(f"End ratio: {training_range.end_ratio}")

In [ ]:
# =============================================================================
# Modeling Configurations
# =============================================================================

MODEL_CONFIGS = {
    "RandomForest": ModelingConfig(
        algorithm="RandomForest",
        params={
            "n_estimators": 100,
            "max_depth": 12,
            "min_samples_leaf": 7,
            "random_state": 42,
            "n_jobs": -1,
        },
    ),
    "CatBoost": ModelingConfig(
        algorithm="CatBoost",
        params={
            "iterations": 100,
            "depth": 12,
            "min_data_in_leaf": 7,
            "learning_rate": 0.1,
            "random_seed": 42,
            "verbose": False,
        },
    ),
}

SELECTED_ALGORITHMS = list(MODEL_CONFIGS)

print("Selected algorithms:")
for algorithm in SELECTED_ALGORITHMS:
    config = MODEL_CONFIGS[algorithm]

    print(f"\n[{algorithm}]")
    for key, value in config.params.items():
        print(f"- {key}: {value}")

In [ ]:
# =============================================================================
# Output Directories
# =============================================================================

OUTPUT_DIR = Path("outputs/05_modeling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALGORITHM_OUTPUT_DIRS = {}

for algorithm in SELECTED_ALGORITHMS:
    algorithm_dir = OUTPUT_DIR / algorithm
    algorithm_dir.mkdir(parents=True, exist_ok=True)
    ALGORITHM_OUTPUT_DIRS[algorithm] = algorithm_dir

print(f"Output directory: {OUTPUT_DIR}")
for algorithm, path in ALGORITHM_OUTPUT_DIRS.items():
    print(f"- {algorithm}: {path}")

In [ ]:
def get_algorithm_output_paths(algorithm):
    algorithm_dir = ALGORITHM_OUTPUT_DIRS[algorithm]

    return {
        "model_bundle": algorithm_dir / "model_bundle.pkl",
        "predictions": algorithm_dir / "lobo_validation_predictions.csv",
        "bearing_metrics": algorithm_dir / "lobo_bearing_metrics.csv",
        "validation_summary": algorithm_dir / "validation_summary.json",
        "calibration_results": algorithm_dir / "calibration_results.json",
        "calibrated_predictions": (
            algorithm_dir / "calibrated_validation_predictions.csv"
        ),
    }


OUTPUT_PATHS = {
    algorithm: get_algorithm_output_paths(algorithm)
    for algorithm in SELECTED_ALGORITHMS
}

In [ ]:
# =============================================================================
# LOBO Model Training
# =============================================================================

model_bundles = {}

for algorithm in SELECTED_ALGORITHMS:
    print(f"Training {algorithm}...")

    model_bundles[algorithm] = train_lobo_models(
        dataset=learning_df,
        feature_cols=FEATURE_COLUMNS,
        target_col=TARGET_COLUMN,
        config=MODEL_CONFIGS[algorithm],
        training_range=training_range,
    )

    bundle = model_bundles[algorithm]

    if len(bundle.models) != len(bundle.fold_metadata):
        raise RuntimeError(
            f"{algorithm}: ModelBundle invariant violated after training: "
            "model/fold counts do not match."
        )

    print(
        f"Completed: {len(bundle.models)} LOBO models, "
        f"{len(bundle.fold_metadata)} folds."
    )

In [ ]:
# =============================================================================
# LOBO Fold Summary
# =============================================================================

fold_rows = []
for algorithm in SELECTED_ALGORITHMS:
    bundle = model_bundles[algorithm]
    for fold in bundle.fold_metadata:
        fold_rows.append({
            "algorithm": algorithm,
            "fold_no": fold.fold_no,
            "held_out_bearing": fold.held_out_bearing,
            "n_train_bearings": len(fold.train_bearings),
            "train_bearings": ", ".join(fold.train_bearings),
        })

fold_summary = pd.DataFrame(fold_rows)
display(fold_summary)

In [ ]:
# =============================================================================
# Prefix / LOBO Bearing Coverage
# =============================================================================

reference_bundle = model_bundles[SELECTED_ALGORITHMS[0]]

lobo_bearings = {
    fold.held_out_bearing
    for fold in reference_bundle.fold_metadata
}

prefix_bearings = set(prefix_definition["bearing"].dropna().unique())

missing_prefix_bearings = sorted(lobo_bearings - prefix_bearings)
unknown_prefix_bearings = sorted(prefix_bearings - lobo_bearings)

if missing_prefix_bearings:
    raise ValueError(
        "Missing prefix definitions for LOBO bearings: "
        f"{missing_prefix_bearings}"
    )

if unknown_prefix_bearings:
    raise ValueError(
        "Prefix definition contains bearings outside "
        f"the LOBO dataset: {unknown_prefix_bearings}"
    )

prefix_coverage = (
    prefix_definition.groupby("bearing")
    .size()
    .rename("n_prefixes")
    .reset_index()
    .sort_values("bearing")
    .reset_index(drop=True)
)

display(prefix_coverage)

In [ ]:
# =============================================================================
# LOBO Prefix Validation Prediction
# =============================================================================

sample_predictions = {}
for algorithm in SELECTED_ALGORITHMS:
    bundle = model_bundles[algorithm]

    predictions = predict_lobo_prefixes(
        bundle=bundle,
        dataset=learning_df,
        prefix_definition=prefix_definition,
    )

    expected_predictions = len(prefix_definition)
    actual_predictions = len(predictions)

    if actual_predictions != expected_predictions:
        raise RuntimeError(
            f"{algorithm}: prediction count mismatch: "
            f"expected {expected_predictions}, "
            f"received {actual_predictions}."
        )

    sample_predictions[algorithm] = predictions

    print(
        f"{algorithm}: "
        f"{actual_predictions} validation predictions"
    )

In [ ]:
# =============================================================================
# Full Validation Prediction
# =============================================================================
FULL_PREDICTION_REQUIRED_COLUMNS = [
    "bearing",
    "elapsed_sec",
    "actual_RUL_norm",
    "predicted_RUL_norm",
]

full_validation_predictions = {}

for algorithm in SELECTED_ALGORITHMS:
    bundle = model_bundles[algorithm]

    predictions = predict_lobo_validation_bearings(
        bundle=bundle,
        dataset=learning_df,
    )

    if predictions.empty:
        raise RuntimeError(
            f"{algorithm}: full validation prediction produced "
            "no prediction rows."
        )

    assert_required_columns(
        predictions,
        FULL_PREDICTION_REQUIRED_COLUMNS,
    )

    expected_validation_bearings = {
        fold.held_out_bearing
        for fold in bundle.fold_metadata
    }

    actual_validation_bearings = set(
        predictions["bearing"].unique()
    )

    if actual_validation_bearings != expected_validation_bearings:
        raise RuntimeError(
            f"{algorithm}: full validation prediction bearing mismatch: "
            f"expected {sorted(expected_validation_bearings)}, "
            f"received {sorted(actual_validation_bearings)}."
        )

    full_validation_predictions[algorithm] = predictions

    print(
        f"{algorithm}: "
        f"{len(predictions)} full validation rows"
    )

In [ ]:
# =============================================================================
# RUL Reconstruction
# =============================================================================

sample_predictions_sec = {}
full_validation_predictions_sec = {}

for algorithm in SELECTED_ALGORITHMS:
    sample_predictions_sec[algorithm] = (
        reconstruct_rul_sec_from_norm(
            df_predictions=sample_predictions[algorithm],
            dataset=learning_df,
            bearing_col="bearing",
            time_col="elapsed_sec",
            pred_norm_col="predicted_RUL_norm",
            actual_norm_col="actual_RUL_norm",
        )
    )

    full_validation_predictions_sec[algorithm] = (
        reconstruct_rul_sec_from_norm(
            df_predictions=full_validation_predictions[algorithm],
            dataset=learning_df,
            bearing_col="bearing",
            time_col="elapsed_sec",
            pred_norm_col="predicted_RUL_norm",
            actual_norm_col="actual_RUL_norm",
        )
    )

In [ ]:
# =============================================================================
# RUL Prediction Trajectories
# =============================================================================

prefix_positions_by_bearing = {}

validation_bearings = sorted(
    full_validation_predictions_sec[
        SELECTED_ALGORITHMS[0]
    ]["bearing"].unique()
)

for bearing in validation_bearings:
    bearing_predictions = (
        full_validation_predictions_sec[
            SELECTED_ALGORITHMS[0]
        ]
        .query("bearing == @bearing")
        .sort_values("elapsed_sec")
        .reset_index(drop=True)
    )

    bearing_prefix = (
        prefix_definition[
            prefix_definition["bearing"] == bearing
        ].sort_values("end_idx")
    )

    positions = []
    for row in bearing_prefix.itertuples(index=False):
        end_idx = int(row.end_idx)

        if end_idx < 0 or end_idx >= len(bearing_predictions):
            raise ValueError(
                f"{bearing}: invalid prefix end_idx={end_idx}. "
                f"Valid range is [0, {len(bearing_predictions) - 1}]."
            )

        actual_rul_norm = (
            bearing_predictions.iloc[end_idx]["actual_RUL_norm"]
        )
        positions.append(1.0 - float(actual_rul_norm))

    prefix_positions_by_bearing[bearing] = positions


for algorithm in SELECTED_ALGORITHMS:
    predictions = full_validation_predictions_sec[algorithm]

    ncols = 2
    nrows = (len(validation_bearings) + ncols - 1) // ncols

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(14, 4.5 * nrows),
        squeeze=False,
    )

    axes = axes.ravel()
    for ax, bearing in zip(axes, validation_bearings):
        plot_rul_prediction(
            df=predictions,
            bearing=bearing,
            prefix_x=prefix_positions_by_bearing[bearing],
            ax=ax,
        )

    for ax in axes[len(validation_bearings):]:
        ax.set_visible(False)

    fig.suptitle(
        f"RUL Prediction Trajectories - {algorithm}",
        fontsize=14,
    )

    fig.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# Prefix Prediction Validation
# =============================================================================

PREDICTION_REQUIRED_COLUMNS = [
    "bearing",
    "end_idx",
    "elapsed_sec",
    "actual_RUL_norm",
    "predicted_RUL_norm",
]

for algorithm in SELECTED_ALGORITHMS:
    predictions = sample_predictions[algorithm]

    assert_required_columns(
        predictions, PREDICTION_REQUIRED_COLUMNS,
    )

    if predictions.empty:
        raise RuntimeError(
            f"{algorithm}: LOBO validation produced no prediction rows."
        )

    prediction_counts = (
        predictions.groupby("bearing")
        .size()
        .rename("n_predictions")
        .reset_index()
        .sort_values("bearing")
        .reset_index(drop=True)
    )

    print(f"{algorithm} prediction counts")
    display(prediction_counts)

In [ ]:
# =============================================================================
# Validation Evaluation
# =============================================================================

global_metrics = {}
bearing_metrics = {}

for algorithm in SELECTED_ALGORITHMS:
    global_result, bearing_result = (
        build_3tier_evaluation_report(
            prediction_df=sample_predictions_sec[algorithm],
            bearing_col="bearing",
            actual_col="actual_RUL_sec",
            predicted_col="predicted_RUL_sec",
        )
    )

    global_metrics[algorithm] = global_result
    bearing_metrics[algorithm] = bearing_result

In [ ]:
# =============================================================================
# Algorithm Comparison
# =============================================================================

validation_comparison = pd.DataFrame(global_metrics).T
validation_comparison.index.name = "Algorithm"

print("Global validation comparison")
display(validation_comparison)

In [ ]:
bearing_comparison = None
for algorithm in SELECTED_ALGORITHMS:
    current = bearing_metrics[algorithm].copy()

    metric_columns = [
        "bearing",
        "n_samples",
        "RMSE",
        "MAE",
        "SMAPE",
        "R2",
        "PHM_Score",
    ]

    current = current[metric_columns]

    if bearing_comparison is None:
        bearing_comparison = current.copy()
        bearing_comparison = bearing_comparison.rename(
            columns={
                column: f"{column}_{algorithm}"
                for column in metric_columns
                if column != "bearing"
            }
        )
    else:
        current = current.rename(
            columns={
                column: f"{column}_{algorithm}"
                for column in metric_columns
                if column != "bearing"
            }
        )

        bearing_comparison = bearing_comparison.merge(
            current,
            on="bearing",
            how="inner",
        )

print("Bearing-level algorithm comparison")
display(bearing_comparison)

In [ ]:
# =============================================================================
# Final Validation Predictions & Artifact Saving
# =============================================================================

PREDICTION_OUTPUT_COLUMNS = [
    "bearing",
    "prefix_id",
    "prefix_no",
    "prefix_ratio",
    "end_idx",
    "elapsed_sec",
    "actual_RUL_norm",
    "predicted_RUL_norm",
    "actual_RUL_sec",
    "predicted_RUL_sec",
    "absolute_error_sec",
]

final_predictions = {}

for algorithm in SELECTED_ALGORITHMS:
    predictions = sample_predictions_sec[algorithm]

    assert_required_columns(
        predictions,
        PREDICTION_OUTPUT_COLUMNS,
    )

    result = predictions[PREDICTION_OUTPUT_COLUMNS].copy()

    if len(result) != len(prefix_definition):
        raise RuntimeError(
            f"{algorithm}: final prediction row count does not match "
            "the number of prefix scenarios."
        )

    final_predictions[algorithm] = result

In [ ]:
# =============================================================================
# Save Model & Validation Artifacts
# =============================================================================
BEARING_METRICS_COLUMNS = [
    "bearing",
    "n_samples",
    "RMSE",
    "MAE",
    "SMAPE",
    "R2",
    "PHM_Score",
]

for algorithm in SELECTED_ALGORITHMS:
    paths = OUTPUT_PATHS[algorithm]

    joblib.dump(
        model_bundles[algorithm],
        paths["model_bundle"],
    )

    final_predictions[algorithm].to_csv(
        paths["predictions"],
        index=False,
    )

    bearing_metrics[algorithm][
        BEARING_METRICS_COLUMNS
    ].to_csv(
        paths["bearing_metrics"],
        index=False,
    )

    validation_summary = {
        metric_name: (
            None if pd.isna(metric_value)
            else float(metric_value)
        )
        for metric_name, metric_value
        in global_metrics[algorithm].items()
    }

    with open(paths["validation_summary"], "w", encoding="utf-8") as file:
        json.dump(
            validation_summary,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print(f"{algorithm} artifacts saved.")

In [ ]:
# =============================================================================
# Feature Importance by Algorithm
# =============================================================================

feature_importance_by_algorithm = {}
for algorithm in SELECTED_ALGORITHMS:
    bundle = model_bundles[algorithm]

    importance_rows = []

    for model, fold in zip(bundle.models, bundle.fold_metadata):
        importance = extract_feature_importance(
            model, bundle.feature_columns,
        )

        if importance is None:
            continue

        importance_rows.append(
            importance.rename(fold.held_out_bearing)
        )

    feature_importance_by_bearing = pd.DataFrame(importance_rows).T
    feature_importance_by_bearing.columns.name = "bearing"

    feature_importance_by_algorithm[
        algorithm] = feature_importance_by_bearing

    print(f"{algorithm} feature importance")
    display(feature_importance_by_bearing)

In [ ]:
# =============================================================================
# Feature Importance Summary
# =============================================================================

feature_importance_summary = {}
for algorithm in SELECTED_ALGORITHMS:
    importance = feature_importance_by_algorithm[algorithm]

    summary = pd.DataFrame({
        "Mean": importance.mean(axis=1),
        "Std": importance.std(axis=1),
        "Min": importance.min(axis=1),
        "Max": importance.max(axis=1),
    }).sort_values("Mean", ascending=False)

    feature_importance_summary[algorithm] = summary

    print(f"{algorithm} feature importance summary")
    display(summary)

# Calibration Experiment

The calibration experiment is an experimental post-processing step.
It does not modify the trained models or the official Stage 05 artifacts.

For each LOBO model, the calibration parameter is fitted independently
using the same life-ratio points defined by `prefix_definition`.
The primary objective is configurable and is evaluated in RUL seconds.

Supported methods:
- `power`: `f(y) = y ** gamma`
- `rational`: `f(y) = a*y / (1 + (a - 1)*y)`

The calibration is point-wise and does not impose temporal monotonicity.


In [ ]:
# =============================================================================
# Calibration Configuration
# =============================================================================

CALIBRATION_METHOD = "rational"
CALIBRATION_OBJECTIVE = "RMSE"

VALID_CALIBRATION_METHODS = {
    None,
    "power",
    "rational",
    "both",
}

if CALIBRATION_METHOD not in VALID_CALIBRATION_METHODS:
    raise ValueError(
        "Unsupported CALIBRATION_METHOD: "
        f"{CALIBRATION_METHOD!r}. "
        "Expected None, 'power', 'rational', or 'both'."
    )

if CALIBRATION_OBJECTIVE not in SUPPORTED_CALIBRATION_OBJECTIVES:
    raise ValueError(
        "Unsupported CALIBRATION_OBJECTIVE: "
        f"{CALIBRATION_OBJECTIVE!r}. "
        f"Expected one of {SUPPORTED_CALIBRATION_OBJECTIVES}."
    )

if CALIBRATION_METHOD is None:
    calibration_methods = []
elif CALIBRATION_METHOD == "both":
    calibration_methods = list(SUPPORTED_CALIBRATION_METHODS)
else:
    calibration_methods = [CALIBRATION_METHOD]

OBJECTIVE_MAXIMIZE = CALIBRATION_OBJECTIVE in {"R2", "PHM_Score"}

print(f"Calibration methods : {calibration_methods or 'SKIP'}")
print(f"Primary objective   : {CALIBRATION_OBJECTIVE}")
print(f"Objective direction : {'maximize' if OBJECTIVE_MAXIMIZE else 'minimize'}")


In [ ]:
# =============================================================================
# Fit Model-specific Calibration Parameters
# =============================================================================

calibration_parameters = {}

for algorithm in SELECTED_ALGORITHMS:
    bundle = model_bundles[algorithm]
    predictions = sample_predictions_sec[algorithm]

    algorithm_parameters = {}

    for method in calibration_methods:
        parameters = {}

        for model_index, fold in enumerate(bundle.fold_metadata):
            bearing = fold.held_out_bearing

            bearing_predictions = predictions[
                predictions["bearing"] == bearing
            ].copy()

            if bearing_predictions.empty:
                raise RuntimeError(
                    "No prefix predictions found for calibration: "
                    f"{algorithm} / {bearing}"
                )

            objective_fn = (
                lambda y_true, y_pred:
                evaluate_predictions(
                    y_true, y_pred,
                )[CALIBRATION_OBJECTIVE]
            )

            parameter = fit_rul_calibration(
                y_true_sec=bearing_predictions[
                    "actual_RUL_sec"
                ].to_numpy(dtype=float),
                predicted_norm=bearing_predictions[
                    "predicted_RUL_norm"
                ].to_numpy(dtype=float),
                method=method,
                prediction_sec_fn=(
                    lambda calibrated_norm,
                    ft=bearing_predictions[
                        "failure_time"
                    ].to_numpy(dtype=float):
                    calibrated_norm * ft
                ),
                objective_fn=objective_fn,
                maximize=OBJECTIVE_MAXIMIZE,
            )

            parameters[model_index] = parameter

        algorithm_parameters[method] = parameters

    calibration_parameters[algorithm] = algorithm_parameters

In [ ]:
for algorithm in SELECTED_ALGORITHMS:
    print(f"\n[{algorithm}]")
    bundle = model_bundles[algorithm]

    for method, parameters in calibration_parameters[
        algorithm].items():
        print(f"\n{method}")

        for model_index, parameter in parameters.items():
            bearing = bundle.fold_metadata[
                model_index].held_out_bearing

            parameter_name = (
                "gamma" if method == "power"
                else "a"
            )

            print(
                f"  model_{model_index} "
                f"({bearing}): "
                f"{parameter_name} = {parameter:.8g}"
            )

In [ ]:
# =============================================================================
# Apply Calibration to Prefix and Full Validation Predictions
# =============================================================================

calibrated_prefix_predictions = {}
calibrated_full_predictions = {}

for algorithm in SELECTED_ALGORITHMS:
    prefix_predictions = (
        sample_predictions_sec[algorithm].copy()
    )
    full_predictions = (
        full_validation_predictions_sec[algorithm].copy()
    )

    bundle = model_bundles[algorithm]

    model_index_by_bearing = {
        fold.held_out_bearing: model_index
        for model_index, fold
        in enumerate(bundle.fold_metadata)
    }

    for method in calibration_methods:
        prefix_norm_column = (
            f"predicted_RUL_norm_{method}"
        )
        prefix_sec_column = (
            f"predicted_RUL_sec_{method}"
        )

        prefix_predictions[prefix_norm_column] = np.nan
        prefix_predictions[prefix_sec_column] = np.nan

        full_predictions[prefix_norm_column] = np.nan
        full_predictions[prefix_sec_column] = np.nan

        for bearing, model_index in model_index_by_bearing.items():
            parameter = calibration_parameters[
                algorithm
            ][method][model_index]

            prefix_mask = (
                prefix_predictions["bearing"] == bearing
            )

            prefix_norm = apply_rul_calibration(
                prefix_predictions.loc[
                    prefix_mask,
                    "predicted_RUL_norm",
                ].to_numpy(dtype=float),
                method=method,
                parameter=parameter,
            )

            prefix_predictions.loc[
                prefix_mask,
                prefix_norm_column,
            ] = prefix_norm

            prefix_predictions.loc[
                prefix_mask,
                prefix_sec_column,
            ] = (
                prefix_norm
                * prefix_predictions.loc[
                    prefix_mask,
                    "failure_time",
                ].to_numpy(dtype=float)
            )

            full_mask = (
                full_predictions["bearing"] == bearing
            )

            full_norm = apply_rul_calibration(
                full_predictions.loc[
                    full_mask,
                    "predicted_RUL_norm",
                ].to_numpy(dtype=float),
                method=method,
                parameter=parameter,
            )

            full_predictions.loc[
                full_mask,
                prefix_norm_column,
            ] = full_norm

            full_predictions.loc[
                full_mask,
                prefix_sec_column,
            ] = (
                full_norm
                * full_predictions.loc[
                    full_mask,
                    "failure_time",
                ].to_numpy(dtype=float)
            )

    calibrated_prefix_predictions[algorithm] = (
        prefix_predictions
    )
    calibrated_full_predictions[algorithm] = (
        full_predictions
    )

print("Calibration prediction generation completed.")

In [ ]:
# =============================================================================
# Raw vs Calibrated Evaluation
# =============================================================================

calibration_global_metrics = {}
calibration_bearing_metrics = {}

for algorithm in SELECTED_ALGORITHMS:
    predictions = calibrated_prefix_predictions[algorithm]

    algorithm_global_metrics = {"Raw": global_metrics[algorithm]}
    algorithm_bearing_metrics = {"Raw": bearing_metrics[algorithm].copy()}

    for method in calibration_methods:
        global_result, bearing_result = (
            build_3tier_evaluation_report(
                prediction_df=predictions,
                bearing_col="bearing",
                actual_col="actual_RUL_sec",
                predicted_col=(
                    f"predicted_RUL_sec_{method}"
                ),
            )
        )

        algorithm_global_metrics[method.capitalize()] = global_result
        algorithm_bearing_metrics[method.capitalize()] = bearing_result

    calibration_global_metrics[algorithm] = algorithm_global_metrics
    calibration_bearing_metrics[algorithm] = algorithm_bearing_metrics

In [ ]:
# =============================================================================
# Calibration Comparison
# =============================================================================

for algorithm in SELECTED_ALGORITHMS:
    comparison = pd.DataFrame(calibration_global_metrics[algorithm]).T
    comparison.index.name = "Method"

    print(f"Global calibration comparison - {algorithm}")
    display(comparison)

In [ ]:
# =============================================================================
# Calibration Experimental Outputs
# =============================================================================

for algorithm in SELECTED_ALGORITHMS:
    paths = OUTPUT_PATHS[algorithm]
    bundle = model_bundles[algorithm]

    calibration_summary = {
        "algorithm": algorithm,
        "method": CALIBRATION_METHOD,
        "primary_objective": CALIBRATION_OBJECTIVE,
        "objective_direction": (
            "maximize" if OBJECTIVE_MAXIMIZE
            else "minimize"
        ),
        "parameters": {},
        "global_metrics": {},
    }

    for method, parameters in calibration_parameters[algorithm].items():
        parameter_name = (
            "gamma" if method == "power"
            else "a"
        )

        calibration_summary["parameters"][method] = {
            str(model_index): {
                "bearing": bundle.fold_metadata[
                    model_index
                ].held_out_bearing,
                parameter_name: float(parameter),
            }
            for model_index, parameter
            in parameters.items()
        }

    for method, metrics in calibration_global_metrics[algorithm].items():
        calibration_summary["global_metrics"][method] = {
            metric: (
                None if pd.isna(value)
                else float(value)
            )
            for metric, value in metrics.items()
        }

    with open(paths["calibration_results"], "w", encoding="utf-8") as file:
        json.dump(
            calibration_summary,
            file,
            ensure_ascii=False,
            indent=2,
        )

    if calibration_methods:
        calibrated_columns = [
            "bearing",
            "prefix_id",
            "prefix_no",
            "prefix_ratio",
            "end_idx",
            "elapsed_sec",
            "actual_RUL_norm",
            "predicted_RUL_norm",
            "actual_RUL_sec",
        ]

        for method in calibration_methods:
            calibrated_columns.extend([
                f"predicted_RUL_norm_{method}",
                f"predicted_RUL_sec_{method}",
            ])

        calibrated_prefix_predictions[
            algorithm][calibrated_columns].to_csv(
            paths["calibrated_predictions"],
            index=False,
        )

    print(f"{algorithm} calibration artifacts saved.")

In [ ]:
# =============================================================================
# Calibration Trajectory Visualization
# =============================================================================

if calibration_methods:
    for algorithm in SELECTED_ALGORITHMS:
        predictions = calibrated_full_predictions[algorithm]

        ncols = 2
        nrows = (len(validation_bearings) + ncols - 1) // ncols

        fig, axes = plt.subplots(
            nrows=nrows,
            ncols=ncols,
            figsize=(14, 4.5 * nrows),
            squeeze=False,
        )

        axes = axes.ravel()

        for ax, bearing in zip(axes, validation_bearings):
            bearing_df = predictions[
                predictions["bearing"] == bearing
            ].sort_values("elapsed_sec")

            x = 1.0 - bearing_df["actual_RUL_norm"].to_numpy(dtype=float)

            ax.plot(
                x,
                bearing_df[
                    "actual_RUL_sec"
                ].to_numpy(dtype=float),
                label="Actual RUL",
            )

            ax.plot(
                x,
                bearing_df[
                    "predicted_RUL_sec"
                ].to_numpy(dtype=float),
                label="Raw",
            )

            for method in calibration_methods:
                ax.plot(
                    x,
                    bearing_df[
                        f"predicted_RUL_sec_{method}"
                    ].to_numpy(dtype=float),
                    label=method.capitalize(),
                )

            for index, prefix_x in enumerate(
                prefix_positions_by_bearing[bearing]):
                ax.axvline(
                    float(prefix_x),
                    linestyle="--",
                    alpha=0.7,
                    label=(
                        "Prefix boundary"
                        if index == 0
                        else None
                    ),
                )

            ax.set_xlabel("life_ratio")
            ax.set_ylabel("RUL_sec")
            ax.set_title(f"RUL Calibration Comparison - {bearing}")
            ax.grid(True)
            ax.legend()

        for ax in axes[len(validation_bearings):]:
            ax.set_visible(False)

        fig.suptitle(
            f"RUL Calibration Comparison - {algorithm}",
            fontsize=14,
        )

        fig.tight_layout()
        plt.show()
else:
    print("Calibration trajectory visualization skipped.")

In [ ]:
# =============================================================================
# Calibration Checkpoint
# =============================================================================

for algorithm in SELECTED_ALGORITHMS:
    paths = OUTPUT_PATHS[algorithm]

    if not paths["calibration_results"].exists():
        raise RuntimeError(
            f"{algorithm}: calibration summary was not created: "
            f"{paths['calibration_results']}"
        )

    if (
        calibration_methods
        and not paths["calibrated_predictions"].exists()
    ):
        raise RuntimeError(
            f"{algorithm}: calibrated prediction artifact "
            "was not created: "
            f"{paths['calibrated_predictions']}"
        )

print("Calibration experiments completed successfully.")

In [ ]:
# =============================================================================
# Stage 05 Final Output Checkpoint
# =============================================================================

required_output_names = [
    "model_bundle",
    "predictions",
    "bearing_metrics",
    "validation_summary",
]

missing_outputs = []

for algorithm in SELECTED_ALGORITHMS:
    for output_name in required_output_names:
        path = OUTPUT_PATHS[algorithm][output_name]

        if not path.exists():
            missing_outputs.append((algorithm, path))

if missing_outputs:
    raise RuntimeError(f"Missing output files: {missing_outputs}")

print("Stage 05 completed successfully.")
print("\nGenerated outputs:")

for algorithm in SELECTED_ALGORITHMS:
    print(f"\n[{algorithm}]")

    for output_name in required_output_names:
        print(f"- {OUTPUT_PATHS[algorithm][output_name].name}")